# 04 · Build recommender (enemy lineup and enemy builds)
For a hero at a moment of the game, rank candidate next items by how much they raise the win chance, given the enemy heroes **and what the enemy team has bought so far** (their spend split across weapon, vitality and spirit, and the specific common items they hold).
Model: `broadway_ml/models/builds.py`. Context terms (net worth gap, minute) absorb "rich players buy big items and win"; only the item terms are exported.
Two checks: does adding item terms beat the context-only model on held-out log loss, and the **pairwise check**: among held-out purchases by the same hero in the same 5-minute window, how often did the winner's item score higher (0.5 = no signal).

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from broadway_ml.flatten import load_tables, item_catalog
from broadway_ml import features as fe, evaluate as ev, export as ex
T = load_tables('../data/tables')
{k: len(v) for k, v in T.items()}
from broadway_ml.models import draft, live, builds, playstyle, rank
import os; os.makedirs('../models', exist_ok=True)

In [ ]:
rep = builds.train(T, '../models')
print(rep['items'], 'items |', rep['val_note'])
rep['comparison']

In [ ]:
pd.DataFrame({'full model': rep['pairwise_full'], 'item only': rep['pairwise_item_only']})

In [ ]:
pd.DataFrame(rep['favoured_vs_enemy_spend'])  # items the model likes more as the enemy leans into each damage type

**Try it:** recommendations for one hero against one enemy setup.

In [ ]:
import json
mdl = json.load(open('../models/build_recommender.json'))
names = item_catalog().set_index('item_id')['name'].to_dict()
def score(item, hero, minute, enemy_heroes, enemy_items, spend):
    r = mdl['rows'][str(item)]; tot = sum(spend)
    z = r['b'] + r['t'] * minute / 30 + r['h'].get(str(hero), 0) + sum(r['e'].get(str(e), 0) for e in enemy_heroes)
    z += sum(w * s / tot for w, s in zip(r['s'], spend)) if tot else 0
    return z + sum(r['v'].get(str(e), 0) for e in enemy_items)
hero = int(T['players'].hero_id.value_counts().index[0])
enemies = [int(h) for h in T['players'].hero_id.value_counts().index[1:7]]
for spend in ([6000, 1000, 1000], [1000, 1000, 6000]):
    top = sorted(mdl['rows'], key=lambda i: -score(i, hero, 18, enemies, [], spend))[:5]
    print('enemy spend (weapon, vitality, spirit) =', spend, '->', [names.get(int(i), i) for i in top])